# Particle identification

This notebook introduces supervised classification for particle identification in a neutrino-detector setting. We will prepare variable-length track data, train logistic-regression and decision-tree classifiers, and evaluate their predictions on held-out events.


## Prerequisites


The first cell verifies the public teaching dataset and plotting helpers, downloads a clean copy when either file is missing or incomplete, then imports the required packages. Downloads are pinned to the tested repository revision.


In [ ]:
from pathlib import Path
import hashlib
import pickle
import shutil
import time
from urllib.request import Request, urlopen

resources = {
    "modules.py": {
        "url": "https://raw.githubusercontent.com/saulam/neutrinoml/10e610a3bf7a628c38fd3ece00e67e195e0b5ca6/modules.py",
        "size": 10041,
        "sha256": "faebdba3b213e6ebec48f99d573cb582ccc08f008ce996dbc852b141e262d190",
    },
    "df_pgun_teaching.p": {
        "url": "https://raw.githubusercontent.com/saulam/neutrinoml/10e610a3bf7a628c38fd3ece00e67e195e0b5ca6/df_pgun_teaching.p",
        "size": 79089990,
        "sha256": "98d4f7fced6a1c9c501d8e468dd20ddc1cec23878d2dc8323a95f4ff20eb98d4",
    },
}


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def ensure_resource(filename, metadata, attempts=3):
    path = Path(filename)
    if (
        path.is_file()
        and path.stat().st_size == metadata["size"]
        and file_sha256(path) == metadata["sha256"]
    ):
        return

    temporary_path = path.with_name(f".{path.name}.download")
    last_error = None

    for attempt in range(attempts):
        try:
            request = Request(metadata["url"], headers={"User-Agent": "neutrino-ml-teaching"})
            with urlopen(request, timeout=120) as response, temporary_path.open("wb") as output:
                shutil.copyfileobj(response, output, length=1024 * 1024)

            if temporary_path.stat().st_size != metadata["size"]:
                raise OSError("downloaded file has the wrong size")
            if file_sha256(temporary_path) != metadata["sha256"]:
                raise OSError("downloaded file failed its checksum")

            temporary_path.replace(path)
            return
        except Exception as error:
            last_error = error
            temporary_path.unlink(missing_ok=True)
            if attempt + 1 < attempts:
                time.sleep(2**attempt)

    raise RuntimeError(f"Could not obtain a valid copy of {filename}") from last_error


for filename, metadata in resources.items():
    ensure_resource(filename, metadata)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

from modules import plot_event, plot_logistic_regression, plot_params_pid, plot_sigmoid

RANDOM_STATE = 7
PID_TO_LABEL = {2212: 0, 13: 1, 211: 2}
FEATURE_NAMES = [
    "dE/dx first node 1",
    "dE/dx first node 2",
    "dE/dx first node 3",
    "dE/dx node n-4",
    "dE/dx node n-3",
    "dE/dx node n-2",
    "dE/dx node n-1",
    "dE/dx node n",
    "TrkLen",
    "TrkEDepo",
    "TrkDir1",
    "TrkDir2",
]


def build_track_features(frame):
    # Return eight ordered dE/dx features and four global track features.
    features = np.full((len(frame), 12), -1.0, dtype=np.float64)

    for row_number, event in enumerate(frame.itertuples(index=False)):
        order = np.asarray(event.NodeOrder, dtype=int)
        dedx = np.asarray(event.Nodededx, dtype=np.float64)[order]

        first = dedx[:3]
        tail = dedx[len(first):][-5:]
        features[row_number, :len(first)] = first
        features[row_number, 8 - len(tail):8] = tail
        features[row_number, 8:] = (
            event.TrkLen,
            event.TrkEDepo,
            event.TrkDir1,
            event.TrkDir2,
        )

    return features


def plot_feature_distributions(features, labels, feature_names, class_names):
    # Compare class-conditional feature distributions.
    fig, axes = plt.subplots(4, 3, figsize=(15, 12))
    for feature_number, axis in enumerate(axes.flat):
        for class_number, class_name in enumerate(class_names):
            values = features[labels == class_number, feature_number]
            if feature_number < 8:
                values = values[values != -1]
            axis.hist(values, bins=50, histtype="step", density=True, label=class_name)
        axis.set_xlabel(feature_names[feature_number])
        axis.set_ylabel("density")
        axis.legend()
    fig.tight_layout()
    plt.show()


def evaluation_table(y_true, y_pred, class_names):
    # Build a compact precision/recall/F1 table.
    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    columns = ["precision", "recall", "f1-score", "support"]
    table = pd.DataFrame(report).T.loc[
        [*class_names, "macro avg", "weighted avg"], columns
    ]
    return table.loc[
        [*class_names, "macro avg", "weighted avg"], columns
    ].round(3)


def show_evaluation(y_true, y_pred, class_names):
    # Display metrics and a conventionally oriented confusion matrix.
    display(evaluation_table(y_true, y_pred, class_names))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        display_labels=class_names,
        cmap="Blues",
        colorbar=False,
        values_format="d",
    )
    plt.show()


## Dataset


Load the event-level DataFrame from the downloaded pickle file.


In [ ]:
with Path("df_pgun_teaching.p").open("rb") as stream:
    df = pickle.load(stream, encoding="latin1")


The dataset contains 59,578 simulated particle-gun events. Its columns are:

- **TruePID**: PDG particle identifier: 2212 (proton), 13 (muon), or 211 (charged pion).
- **TrueMomentum**: generated particle momentum in MeV/$c$.
- **NNodes**: number of reconstructed 3D points in the event.
- **NodeOrder**: integer indices that order the reconstructed nodes along the track.
- **NodePosX**, **NodePosY**, **NodePosZ**: node coordinates in mm.
- **NodeT**: time-like node coordinate in the dataset's native units.
- **Nodededx**: per-node energy-loss proxy in the dataset's native units.
- **TrkLen**: reconstructed track length in mm.
- **TrkEDepo**: total deposited-energy proxy in the dataset's native units.
- **TrkDir1**, **TrkDir2**: two dimensionless track-direction components.

The direction columns are bounded near $[-1, 1]$; they are not angles in degrees.


In [ ]:
df.head()


A correlation matrix is meaningful only for scalar numerical features. The array-valued node columns are excluded, and `TruePID` is omitted because a numerical correlation of arbitrary PDG codes is not physically meaningful.


In [ ]:
scalar_features = [
    "TrueMomentum", "NNodes", "TrkLen", "TrkEDepo", "TrkDir1", "TrkDir2"
]
df[scalar_features].corr().round(3)


Particle trajectories can be represented either by detector hits or by fitted nodes. Hits correspond to detector elements with recorded energy, whereas nodes are positions produced by track reconstruction.

<div>
<img src="https://raw.githubusercontent.com/saulam/neutrinoml/main/hit.png" width="400"/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img src="https://raw.githubusercontent.com/saulam/neutrinoml/main/node.png" width="400"/>
</div>


The next plot shows the reconstructed nodes of one event in detector coordinates. Change `event_number` to inspect a different event.


In [ ]:
event_number = 5
plot_event(df, event_number)


A supervised classifier learns a label $y$ from a fixed-size feature vector $\mathbf{x}$. Track events contain different numbers of nodes, so we begin with two fixed-size global features: `TrkLen` and `TrkEDepo`.

For the first binary problem, keep only protons and muons and encode them as 0 and 1. Pions must be removed before constructing the feature matrix; otherwise their empty rows would be mislabeled as protons.


In [ ]:
binary_df = df[df["TruePID"].isin([2212, 13])].reset_index(drop=True)
X2 = binary_df[["TrkLen", "TrkEDepo"]].to_numpy(dtype=np.float64)
y2 = binary_df["TruePID"].map({2212: 0, 13: 1}).to_numpy(dtype=int)

pd.Series(y2).map({0: "proton", 1: "muon"}).value_counts().rename("events")


Before fitting a model, inspect the two features. The overlap between the class distributions indicates where classification will be difficult.


In [ ]:
plot_params_pid(
    X2,
    y2,
    param_names=["track length [mm]", "deposited-energy proxy"],
    y_names=["proton", "muon"],
)


The classes occupy visibly different, but overlapping, regions of this feature space.


## Logistic regression


We reserve 40% of the events for testing and stratify the split so that both subsets retain the class proportions. The test set must remain untouched while the model and preprocessing are fitted.

`StandardScaler` is fitted on the training features only, then the learned transformation is applied to the test features. Fitting the scaler before splitting would leak information from the test set into training. For a real model-selection exercise, use a validation set or cross-validation within the training set and evaluate the test set only once.


In [ ]:
X2_train_raw, X2_test_raw, y2_train, y2_test = train_test_split(
    X2,
    y2,
    test_size=0.4,
    random_state=RANDOM_STATE,
    stratify=y2,
)

scaler2 = StandardScaler()
X2_train = scaler2.fit_transform(X2_train_raw)
X2_test = scaler2.transform(X2_test_raw)


Logistic regression is a linear classifier and a generalized linear model. For a feature vector $\mathbf{x}$, it first forms the linear score

$$
z = \mathbf{x}^{\mathsf{T}}\mathbf{w} + b,
$$

then converts that score to a probability with the sigmoid function

$$
\sigma(z) = \frac{1}{1 + e^{-z}}.
$$


In [ ]:
plot_sigmoid()


With the usual probability threshold of 0.5, the predicted class is

$$
\hat{y} =
\begin{cases}
0 & \text{if } \sigma(z) < 0.5, \\\\
1 & \text{if } \sigma(z) \geq 0.5.
\end{cases}
$$


For the encoding used here,

$$
P(y=1\mid\mathbf{x}) = \sigma(\mathbf{x}^{\mathsf{T}}\mathbf{w}+b),
\qquad
P(y=0\mid\mathbf{x}) = 1-P(y=1\mid\mathbf{x}).
$$

The model learns $\mathbf{w}$ and $b$ by minimizing regularized log loss. Scikit-learn applies $L_2$ regularization by default.


In [ ]:
log_reg2 = LogisticRegression(max_iter=1000).fit(X2_train, y2_train)

pd.Series(
    [*log_reg2.coef_[0], log_reg2.intercept_[0]],
    index=["weight: standardized TrkLen", "weight: standardized TrkEDepo", "intercept"],
    name="fitted parameter",
).to_frame()


The probability returned by `predict_proba` can be reproduced directly from the fitted parameters. This calculation uses the standardized feature vector because that is the representation on which the model was trained.


In [ ]:
def sigmoid(value):
    return 1.0 / (1.0 + np.exp(-value))


event_number = 0
event_features = X2_train[event_number:event_number + 1]
probability_model = log_reg2.predict_proba(event_features)[0, 1]
logit = event_features @ log_reg2.coef_[0] + log_reg2.intercept_[0]
probability_manual = sigmoid(logit).item()

assert np.isclose(probability_model, probability_manual)

pd.DataFrame(
    {
        "quantity": [
            "model probability for muon",
            "manual probability for muon",
            "predicted class",
            "true class",
        ],
        "value": [
            probability_model,
            probability_manual,
            ["proton", "muon"][log_reg2.predict(event_features).item()],
            ["proton", "muon"][y2_train[event_number]],
        ],
    }
)


The decision boundary below is shown in standardized coordinates, matching the model input. The probability panel shows the model's estimated proton probability.


In [ ]:
plot_logistic_regression(
    log_reg2,
    X2_test,
    y2_test,
    param_names=["standardized TrkLen", "standardized TrkEDepo"],
    y_names=["proton", "muon"],
)


Precision, recall, and $F_1$ score provide class-specific views of performance. In the confusion matrix, rows are true classes and columns are predicted classes.


In [ ]:
y2_pred = log_reg2.predict(X2_test)
show_evaluation(y2_test, y2_pred, ["proton", "muon"])


Two global features already separate many events. We now add local energy-loss information from the reconstructed nodes.


A fixed-size representation can retain up to the first three ordered node values, up to the last five remaining node values, and four global track features.

<div>
<img src="https://raw.githubusercontent.com/saulam/neutrinoml/main/reg.png" width="500"/>
</div>

The resulting vector has 12 features. Tracks with fewer than eight non-overlapping node positions use $-1$ as a missing-value sentinel. Tail values are right-aligned so that the final node always occupies the feature labelled `node n`.


In [ ]:
X12_binary = build_track_features(binary_df)
y12_binary = binary_df["TruePID"].map({2212: 0, 13: 1}).to_numpy(dtype=int)

pd.DataFrame(X12_binary, columns=FEATURE_NAMES).head()


Class-conditional histograms reveal which features may be discriminative. Density normalization makes the distribution shapes comparable even when class counts differ.


In [ ]:
plot_feature_distributions(
    X12_binary,
    y12_binary,
    feature_names=FEATURE_NAMES,
    class_names=["proton", "muon"],
)


Split the unscaled features first, then fit the scaler on the training subset only.


In [ ]:
X12_train_raw, X12_test_raw, y12_train, y12_test = train_test_split(
    X12_binary,
    y12_binary,
    test_size=0.4,
    random_state=RANDOM_STATE,
    stratify=y12_binary,
)

scaler12 = StandardScaler()
X12_train = scaler12.fit_transform(X12_train_raw)
X12_test = scaler12.transform(X12_test_raw)


Fit logistic regression to the 12 standardized features.


In [ ]:
log_reg12 = LogisticRegression(max_iter=1000).fit(X12_train, y12_train)
y12_pred = log_reg12.predict(X12_test)

show_evaluation(y12_test, y12_pred, ["proton", "muon"])


The additional node-level information improves the binary classifier. Logistic regression also has a multinomial extension for problems with more than two classes; later we use a decision tree to include pions.


## Decision trees


A decision tree recursively partitions feature space. Each internal node applies a threshold to one feature, and each leaf stores a class prediction. A shallow tree is easy to inspect, although increasing depth can overfit the training data.

Tree splits are invariant to monotonic rescaling, so this model is trained on the unscaled features. The displayed thresholds therefore remain in the original feature units. We start with a maximum depth of 3.


In [ ]:
binary_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=3,
    random_state=RANDOM_STATE,
).fit(X12_train_raw, y12_train)

y12_tree_pred = binary_tree.predict(X12_test_raw)


Evaluate the tree on the held-out test events.


In [ ]:
show_evaluation(y12_test, y12_tree_pred, ["proton", "muon"])


The tree diagram shows the learned thresholds, class proportions, and prediction at each node.


In [ ]:
plt.figure(figsize=(24, 10))
plot_tree(
    binary_tree,
    feature_names=FEATURE_NAMES,
    class_names=["proton", "muon"],
    filled=True,
    rounded=True,
    proportion=True,
    precision=2,
    fontsize=9,
)
plt.tight_layout()
plt.show()


Next, construct the same 12 features for all three particle species. This is a multiclass problem with proton, muon, and pion labels.


In [ ]:
X12_multiclass = build_track_features(df)
y12_multiclass = df["TruePID"].map(PID_TO_LABEL).to_numpy(dtype=int)

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X12_multiclass,
    y12_multiclass,
    test_size=0.4,
    random_state=RANDOM_STATE,
    stratify=y12_multiclass,
)


Inspect the three class-conditional feature distributions.


In [ ]:
plot_feature_distributions(
    X12_multiclass,
    y12_multiclass,
    feature_names=FEATURE_NAMES,
    class_names=["proton", "muon", "pion"],
)


Fit the same depth-3 tree to the three-class training set.


In [ ]:
multiclass_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=3,
    random_state=RANDOM_STATE,
).fit(X3_train, y3_train)

y3_pred = multiclass_tree.predict(X3_test)
show_evaluation(y3_test, y3_pred, ["proton", "muon", "pion"])


The three-class task is harder, especially where pion signatures overlap those of protons or muons. A deeper tree can lower training error but may worsen generalization. Choose `max_depth` with cross-validation on the training set rather than repeatedly selecting it from test-set performance.


## Exercises


Try to improve the classifier while preserving an unbiased final evaluation:

1. Keep the test set fixed and use cross-validation on the training set for all model and hyperparameter choices.
2. Compare alternative feature representations motivated by particle energy loss and track geometry.
3. Tune the regularization strength `C` for logistic regression and `max_depth`, `min_samples_leaf`, or `ccp_alpha` for the decision tree.
4. Compare with a [support-vector classifier](https://scikit-learn.org/stable/modules/svm.html), [naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html), or a [random forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html).
5. Report precision, recall, $F_1$, and the confusion matrix, not only overall accuracy.
